# Google Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/google_reviews.py`.

Covers:
1. **Playwright scraping** (Plan A) – fetches reviews from Google Maps, returns both Google and TripAdvisor reviews
2. **API call** (Plan B) – fetching reviews via Google Places API (max 5 reviews)
3. Ollama classification – topic & sentiment detection
4. Manual review input – adding reviews that scrapers can't fetch

**Requirements:** Playwright installed, Ollama running locally with `qwen2.5:7b`. `GOOGLE_MAPS_API_KEY` for API fallback.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [2]:
from src.sites import google_reviews as gr
from src.classification import classify_review, is_ollama_available, _parse_classification
import requests

api_key = os.getenv('GOOGLE_MAPS_API_KEY')

print(f'Hotel: {gr.ANANEA_HOTEL}')
print(f'Google query: {gr.ANANEA_GOOGLE_QUERY}')
print(f'API key: {"set" if api_key else "NOT SET"}')

Hotel: Ananea Castelo Suites Hotel
Google query: Ananea Castelo Suites Algarve, Portugal
API key: NOT SET


## 1. Playwright – Fetch Reviews from Google Maps (Plan A)

Scrapes reviews directly from Google Maps using Playwright. This returns both
**Google** and **TripAdvisor** reviews — each tagged with `source_platform`.
No API key required.

In [3]:
from src.sites.google_scraper import google_get_reviews_playwright, GOOGLE_MAPS_URLS

# Get the Google Maps URL for Ananea
maps_url = GOOGLE_MAPS_URLS.get(gr.ANANEA_HOTEL)
print(f'Google Maps URL: {maps_url}')
print()

# Fetch reviews via Playwright (both Google and TripAdvisor)
pw_reviews = google_get_reviews_playwright(maps_url, max_reviews=50)
print(f'Total reviews fetched: {len(pw_reviews)}')

# Count by platform
google_count = sum(1 for r in pw_reviews if r.get('source_platform') == 'google')
ta_count = sum(1 for r in pw_reviews if r.get('source_platform') == 'tripadvisor')
print(f'  Google reviews: {google_count}')
print(f'  TripAdvisor reviews: {ta_count}')
print()

for r in pw_reviews[:10]:
    platform = r.get('source_platform', '?')
    icon = 'G' if platform == 'google' else 'TA'
    print(f'  [{icon}] {r["rating"]}/5 by {r["author_name"]} ({r["published_date"]}) – {r["text"][:80]}...')

Google Maps URL: https://maps.app.goo.gl/QsTaS8vLupyrC3hQ8

Total reviews fetched: 50
  Google reviews: 37
  TripAdvisor reviews: 13

  [G] 5.0/5 by breyner sanchez () – ...
  [G] 5.0/5 by Anne b (2026-01-13) – Everything was perfect: the cleanliness, the setting, the attentive staff, and t...
  [G] 5.0/5 by Anita (2026-01-13) – Go without hesitation! The location is very quiet, and everyone is very kind and...
  [G] 5.0/5 by Alex Vl (2026-01-13) – Impeccable service. The staff were extremely kind, attentive, and always smiling...
  [G] 5.0/5 by James Akachi Oguledo (2026-01-13) – ...
  [G] 5.0/5 by Gisele Secco (2026-01-13) – We held our Christmas event at Castelo Suites and they were impeccable. The hote...
  [G] 5.0/5 by Rubia Cordero (2025-12-14) – We enjoyed a few days at this beautiful hotel. Modern room and facilities, a lov...
  [G] 5.0/5 by Manolo (2025-12-14) – We stayed during the Constitution Day long weekend and the newly opened hotel wa...
  [G] 5.0/5 by francisco cordero

In [4]:
# Inspect a single Playwright review
if pw_reviews:
    print(json.dumps({k: v for k, v in pw_reviews[1].items() if k != 'id'}, indent=2, ensure_ascii=False))

{
  "rating": 5.0,
  "title": "",
  "text": "Everything was perfect: the cleanliness, the setting, the attentive staff, and the location.",
  "published_date": "2026-01-13",
  "author_name": "Anne b",
  "source_platform": "google",
  "trip_type": "vacation",
  "travel_group": "couple",
  "original_language": "french"
}


## 2. API – Fetch Google Reviews (Plan B / Fallback)

The Google Places API (New) returns a **maximum of 5 reviews** per request
with no pagination. Use this when Playwright is unavailable.

Requires `GOOGLE_MAPS_API_KEY`.

In [4]:
# Fetch reviews for Ananea via Google Places Text Search
raw_reviews = gr.google_get_reviews(gr.ANANEA_GOOGLE_QUERY, api_key=api_key)
print(f'Reviews returned: {len(raw_reviews)}')
print()
for r in raw_reviews:
    rid = gr._extract_review_id(r)
    text = gr._extract_review_text(r)
    date = gr._extract_publish_date(r)
    rating = r.get('rating', 'N/A')
    author = r.get('authorAttribution', {}).get('displayName', 'Unknown')
    lang = r.get('originalText', {}).get('languageCode', r.get('text', {}).get('languageCode', '?'))
    print(f'  [{rid[:20]}...] {rating}/5 lang={lang} by {author} on {date}')
    print(f'    {text[:100]}...')
    print()

Reviews returned: 5

  [Ci9DQUlRQUNvZENodHlj...] 4/5 lang=en by Raj on 2025-11-11
    I booked this hotel on a whim as there were no jet 2 reviews on site. I did take a look at trip advi...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Lorena David on 2025-09-18
    The hotel was amazing! We stayed in the Maretta suite for a week and were very pleased with the room...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Elzbieta Krzyszton on 2025-09-09
    The best  hotel and staff, beautifull rooms and pool, the view, everything is great, i wish come bac...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Izzy Roberts on 2025-09-10
    I loved the violinist that plays in the hotel ! His music has a modern twist like Bridgerton 🎻🎻...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Helen Turner on 2025-10-03
    Wonderful place to stay, lovely breakfast and fabulous staff...



In [5]:
# Inspect raw API response for first review
if raw_reviews:
    print(json.dumps(raw_reviews[0], indent=2, ensure_ascii=False))

{
  "name": "places/ChIJ5acUwUfNGg0RrUqtcamF0Y8/reviews/Ci9DQUlRQUNvZENodHljRjlvT2tkMlkwOVZTRmhJTmpOVFJrZzRTVkZsZGxCeE9FRRAB",
  "relativePublishTimeDescription": "4 months ago",
  "rating": 4,
  "text": {
    "text": "I booked this hotel on a whim as there were no jet 2 reviews on site. I did take a look at trip advisor which had a mixed bag and took the gamble anyway, definitely paid off!\n\nThe hotel is in a great location, they have a shuttle to the beach which I did not use, as I would say to three of the closest beaches it was less than a 20 walk. The room had a great view of the pool area, spacious enough for two people, smart tv which is a nice touch. Iron is available upon request.\n\nPros:\n\n- [ ] Great customer service all around, no issues which is hard to come by.\n- [ ] Plenty of sunbeds available and no rush to secure one like some other hotels\n- [ ] I liked that it was quiet, note this isn’t an adult only hotel, as during my stay there were a family with younger child

## 3. Ollama Classification – Test

Uses **Playwright reviews** (Plan A) if available, otherwise **API reviews** (Plan B).
Reviews are normalised to a common format: `{id, rating, text, author_name, published_date}`.

In [5]:
# Check Ollama availability
ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    # List available models
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

Ollama available: True
Models: ['qwen2.5:7b']


In [6]:
# Build unified review list: Playwright (Plan A) → API (Plan B)
reviews_for_classification = []

# Plan A: use Playwright reviews (already in common format)
if 'pw_reviews' in dir() and pw_reviews:
    # Filter to Google-only reviews for this notebook (TripAdvisor handled separately)
    reviews_for_classification = [r for r in pw_reviews if r.get('source_platform') == 'google']
    print(f'Using {len(reviews_for_classification)} Google reviews from Playwright')

# Plan B: convert API reviews to common format
if not reviews_for_classification and 'raw_reviews' in dir() and raw_reviews:
    for r in raw_reviews:
        reviews_for_classification.append({
            'id': gr._extract_review_id(r),
            'rating': r.get('rating'),
            'text': gr._extract_review_text(r),
            'author_name': (r.get('authorAttribution') or {}).get('displayName', ''),
            'published_date': gr._extract_publish_date(r),
            'source_platform': 'google',
        })
    print(f'Using {len(reviews_for_classification)} reviews from API')

if not reviews_for_classification:
    print('No reviews available for classification')
else:
    print(f'Sample: {reviews_for_classification[0]["author_name"]} – {reviews_for_classification[0]["text"][:80]}...')

Using 37 Google reviews from Playwright
Sample: breyner sanchez – ...


In [7]:
# Classify a single review (pick the first with text)
test_review = next((r for r in reviews_for_classification if r.get('text')), None)
if test_review and ollama_ok:
    print(f'Review by: {test_review["author_name"]}')
    print(f'Text: {test_review["text"][:300]}...')
    print()
    topics = classify_review(test_review['text'])
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('No review with text found or Ollama not available')

Review by: Anne b
Text: Everything was perfect: the cleanliness, the setting, the attentive staff, and the location....

Classification result (2 topics):
  🟢 employees = positive
  🟢 commodities = positive


In [ ]:
# Classify ALL fetched reviews, save to JSON (overwrites existing reviews by ID)
from datetime import datetime

if ollama_ok and reviews_for_classification:
    json_path = str(ROOT / 'data' / 'google_reviews.json')
    existing = gr.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in reviews_for_classification:
        text = r.get('text', '')
        if not text:
            continue
        topics = classify_review(text)

        record = {
            'id': r['id'],
            'hotel': gr.ANANEA_HOTEL,
            'source': 'google',
            'rating': r.get('rating'),
            'title': '',
            'text': text,
            'published_date': r.get('published_date', ''),
            'author_name': r.get('author_name', ''),
            'country': r.get('country', '') or 'Unknown',
            'trip_type': r.get('trip_type', '') or 'Unknown',
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if r['id'] in existing_by_id else 'added'
        existing_by_id[r['id']] = record

        stars = '★' * int(r.get('rating') or 0)
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{stars} [{action}] {r.get('author_name', 'N/A')[:35]:35s} → {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    gr.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available or no reviews – skip classification')

In [ ]:
# Debug: see the classification result for a specific review
# Change the index to test different reviews
DEBUG_INDEX = 0

if ollama_ok and reviews_for_classification:
    review = reviews_for_classification[DEBUG_INDEX]
    text = review.get('text', '')
    print(f'Review by: {review.get("author_name", "Unknown")}')
    print(f'Full text:\n{text}\n')
    print('--- Sending to Ollama ---')

    topics = classify_review(text)
    print(f'\nClassification result ({len(topics)} topics):')
    for t in topics:
        icon = '🟢' if t['sentiment'] == 'positive' else '🔴'
        detail = f" – {t['detail']}" if t.get('detail') else ""
        print(f"  {icon} {t['topic']} = {t['sentiment']}{detail}")
else:
    print('Ollama not available or no reviews')

## 4. Manual Review Input

Since the Google API returns only 5 reviews per request with no pagination,
you can manually add reviews below. Fill in the fields and run the cell.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'author_name': 'John D.',                      # reviewer name (used for ID)
    'rating': 5,                                    # 1-5
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
    'trip_type': 'Couple',                          # Solo, Couple, Family, Friends, Business
    'country': 'Unknown',                           # reviewer country
}
# ========================================

# Generate unique ID from name + date
id_seed = f"{manual_review['author_name']}_{manual_review['published_date']}_{manual_review['text'][:50]}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

record = {
    'id': review_id,
    'hotel': gr.ANANEA_HOTEL,
    'source': 'google',
    'rating': manual_review['rating'],
    'title': '',  # Google reviews have no title
    'text': manual_review['text'],
    'published_date': manual_review['published_date'],
    'author_name': manual_review['author_name'],
    'country': manual_review.get('country', '') or 'Unknown',
    'trip_type': manual_review.get('trip_type', '') or 'Unknown',
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
    'manual': True,
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = classify_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text \u2013 skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'google_reviews.json')
existing = gr.load_reviews(json_path)

# Check for duplicates
existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'\u26a0\ufe0f  Review {record["id"]} already exists \u2013 skipping')
else:
    existing.append(record)
    gr.save_reviews(existing, json_path)
    print(f'\u2705 Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once. Useful for adding reviews from Google Maps
that aren't returned by the API.

In [ ]:
import hashlib
from datetime import datetime

# Add multiple reviews at once
batch_reviews = [
    {
        'author_name': 'Maria S.',
        'rating': 4,
        'text': 'Replace with actual review text...',
        'published_date': '2026-02-15',
        'trip_type': 'Couple',
        'country': 'Unknown',
    },
    # Add more reviews here...
]

json_path = str(ROOT / 'data' / 'google_reviews.json')
existing = gr.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['author_name']}_{mr['published_date']}_{mr['text'][:50]}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

    if rid in existing_ids:
        print(f'⚠️  Skip duplicate: {mr["author_name"]}')
        continue

    rec = {
        'id': rid,
        'hotel': gr.ANANEA_HOTEL,
        'source': 'google',
        'rating': mr['rating'],
        'title': '',  # Google reviews have no title
        'text': mr['text'],
        'published_date': mr['published_date'],
        'author_name': mr['author_name'],
        'country': mr.get('country', '') or 'Unknown',
        'trip_type': mr.get('trip_type', '') or 'Unknown',
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
        'manual': True,
    }

    # Classify if Ollama available
    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = classify_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')

    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"✅ {mr['author_name']} → {topic_str or '(unclassified)'}")

if added:
    gr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [10]:
json_path = str(ROOT / 'data' / 'google_reviews.json')
reviews = gr.load_reviews(json_path)

# Find reviews that are "classified" but have 0 topics \u2013 likely LLM failures
needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
# Also find unclassified reviews
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            author = r.get('author_name', 'N/A')
            print(f"  \u2705 {author[:40]} \u2192 {topic_str or '(still none)'}")
        except Exception as e:
            author = r.get('author_name', 'N/A')
            print(f"  \u274c {author[:40]} \u2192 Error: {e}")

    gr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

Total reviews: 33
Classified with 0 topics (needs retry): 4
Unclassified: 0
  ✅ K IKA → (still none)
  ✅ Larissa Goldnik → (still none)
  ✅ Christophe LEHMANN → employees(pos), commodities(pos)
  ✅ Heger → (still none)

Done. Saved 33 reviews.


## 7. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [8]:
json_path = str(ROOT / 'data' / 'google_reviews.json')
reviews = gr.load_reviews(json_path)

with_text = [r for r in reviews if r.get('text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_review(r['text'])
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '\U0001f504' if old_str != new_str else '\u2705'
            author = r.get('author_name', 'N/A')
            print(f"  {changed} [{i}/{len(with_text)}] {author[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            author = r.get('author_name', 'N/A')
            print(f"  \u274c [{i}/{len(with_text)}] {author[:40]} \u2192 Error: {e}")

    gr.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

Total reviews: 33
Reviews with text (will reclassify): 33
  🔄 [1/33] Raj
       old: employees(pos), commodities(pos), comfort(neg), cleaning(pos), meals(pos), meals(neg), commodities(neg)
       new: employees(pos), commodities(pos), comfort(pos), meals(pos), cleaning(pos), comfort(neg), meals(neg), commodities(neg)
  🔄 [2/33] Lorena David
       old: employees(pos), commodities(pos), commodities(neg), meals(pos), meals(neg)
       new: employees(pos), commodities(pos), comfort(pos), cleaning(pos), commodities(neg), comfort(neg), meals(pos), meals(neg)
  ✅ [3/33] Elzbieta Krzyszton
  🔄 [4/33] Izzy Roberts
       old: employees(pos), commodities(pos)
       new: employees(pos)
  ✅ [5/33] Helen Turner
  🔄 [6/33] Anne b
       old: employees(pos), cleaning(pos), quality_price(pos)
       new: commodities(pos), employees(pos), quality_price(pos)
  🔄 [7/33] Anita
       old: (none)
       new: employees(pos), meals(pos), return(pos)
  🔄 [8/33] Alex Vl
       old: employees(pos), meals(pos)

## 8. Run CLI Scraper

Run the full CLI scraper via `python -m` to test the end-to-end pipeline.

In [ ]:
import subprocess
from datetime import datetime

date_col = datetime.now().strftime('%Y-%m-%d')
cmd = [sys.executable, '-m', 'src.sites.google_reviews',
       '--date', date_col, '--skip-classification']
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print(f'Return code: {result.returncode}')

## 9. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'google_reviews.json')
reviews = gr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if r.get("manual"))}')
print()

# Rating distribution
if reviews:
    ratings = [r.get('rating', 0) for r in reviews if r.get('rating')]
    if ratings:
        print(f'Average rating: {sum(ratings)/len(ratings):.1f}/5')
        for star in range(5, 0, -1):
            count = ratings.count(star)
            bar = '█' * count
            print(f'  {star}★: {bar} ({count})')
        print()

# Country breakdown
country_counts = {}
for r in reviews:
    c = r.get('country', 'Unknown') or 'Unknown'
    country_counts[c] = country_counts.get(c, 0) + 1

if country_counts:
    print('Country breakdown:')
    for country, count in sorted(country_counts.items(), key=lambda x: -x[1]):
        print(f'  {country}: {count}')
    print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')